# 🧪 Experiment Notebook — Rubric & Prompt Iteration
## INFO 290 Group 5

---

### 📌 Context: Where the Paper Stands

Our draft paper (shared alongside this notebook) reports:
- **V2 (RAG+Visual) slightly outperforms V1 (Zero-Shot)** on the test set (4.46 vs 4.42), with the clearest gain in visual specificity (+0.27).
- **V3 (Agentic) does not improve over V2** with Mistral-7B.
- **Two hypotheses** explain why the gains are small:
  - **(a) Model capacity:** Mistral-7B may be too weak to leverage retrieved context. Generator comparison (n=3) shows Gemini 2.5 Flash benefits from RAG (+0.20) while Mistral does not (-0.33).
  - **(b) Rubric sensitivity:** 3 of 5 evaluation dimensions hit ceiling (~5.0), leaving no room to discriminate variants.

### 🎯 What This Notebook Is For

Four planned analyses to strengthen the paper before submission:

| Analysis | What | Section | Time | Paper section updated |
|----------|------|---------|------|----------------------|
| **A. Statistical test** | Wilcoxon signed-rank on V1 vs V2 per-record scores | **5E** (new) | 5 min | §5.7 Test Set Evaluation |
| **B. Rubric tightening** | Re-judge cached outputs with stricter rubric anchors | **5A** | 30 min | §5.8 Ceiling Effect Analysis |
| **C. Generator comparison (n↑)** | Re-run Gemini/GPT-4o-mini on val subset (n=20) | **5B** (swap model) | 1-2 hr | §5.4 Generator Comparison |
| **D. Prompt improvement** | Add craft guidance to prompt, regenerate on val subset | **5B** / **5D** | 1 hr | §5.6 / §5.7 |

### How to edit prompts and rubrics

**Do NOT edit code cells.** Edit these files under `experiment_config/`:

| What | Files | After saving |
|------|-------|--------------|
| Prompt text | `prompt_baseline.txt`, `prompt_trial.txt` | Re-run **Cell 3-F**, then downstream |
| Rubric JSON | `rubric_baseline.json`, `rubric_trial.json` | Re-run **Cells 4-A & 4-B**, then downstream |

Placeholders in prompt files: `__PREMISE__`, `__THEME__`, `__NEGATIVES__`, `__REFS__`

### How to run this notebook

```
Step 1: Run Section 0–2 (Setup, Data, RAG index, Model, Captions)    ~5 min
Step 2: Run Section 3 (Core Pipeline functions)                       instant
Step 3: Run Section 4 (Evaluation functions)                          instant
        → Pipeline ready.

Step 4: EXPERIMENT — Section 5
        5E: Statistical test on existing results (Analysis A) — do this FIRST
        5A: Re-judge cached outputs with new rubric (Analysis B)
        5B: Val subset with new prompt (Analysis D) or new generator (Analysis C)
        5C: One-idea sanity check (eyeball outputs)
        5D: Baseline vs trial side-by-side comparison

Step 5: Section 6 — Final test set evaluation (ONE TIME, after Section 5 is done)
```

### ⚠️ Rules
- **TEST SET IS FROZEN** — all iteration in Section 5 uses val set only
- **Delete `test_results_final.json`** to force a fresh Section 6 run
- **Best config from tuning**: k=3, genre_filter=False, temp=0.6
- **`experiment_config/` must exist** — see "Initial config files" cell below


## Section 0: Setup

In [ ]:
# Cell 0-A: Install dependencies
!pip -q install "transformers>=4.41" "accelerate>=0.30" bitsandbytes \
                 sentence-transformers faiss-cpu pandas numpy requests tqdm \
                 google-generativeai openai

In [ ]:
# Cell 0-B: Imports & credentials
import os, json, re, time
import numpy as np
import pandas as pd
import requests
import torch
import faiss
from tqdm import tqdm
from google.colab import userdata

TMDB_BEARER    = userdata.get("TMDB_BEARER")
HF_TOKEN       = userdata.get("HF_TOKEN")
GH_TOKEN       = userdata.get("GITHUB_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("✅ Imports done")

In [ ]:
# Cell 0-C: Clone repo
REPO_ROOT = "ryuichi-github-info290-2026s-group5"
if os.path.exists(REPO_ROOT):
    !git -C {REPO_ROOT} pull
else:
    !git clone https://{GH_TOKEN}@github.com/ryuichi-github/ryuichi-github-info290-2026s-group5.git

CACHE_DIR = os.path.join(REPO_ROOT, "to_be_cached")
os.makedirs(CACHE_DIR, exist_ok=True)
print("✅ Repo ready")

In [ ]:
# Cell 0-D: Create experiment_config/ with initial files (run once)
# These are the CURRENT prompt and rubric from the main pipeline.
# baseline = frozen reference. trial = your experiments.

import os

EXP_DIR = os.path.join(REPO_ROOT, "experiment_config")
os.makedirs(EXP_DIR, exist_ok=True)

# ── Prompt template (same for baseline and trial initially) ──
PROMPT_TEMPLATE = '''You are a movie marketing generator.

Generate marketing assets for the movie described below.

OUTPUT: ONE JSON object with EXACTLY these keys:
- overview (<= 80 words, compelling promo synopsis)
- tagline (<= 12 words)
- poster_art_direction (<= 60 words)

JSON FORMAT:
{"overview": "", "tagline": "", "poster_art_direction": ""}

MOVIE DETAILS:
Core Premise: __PREMISE__
Thematic Core: __THEME__
IMPORTANT - Do NOT do the following: __NEGATIVES__

REFERENCES from similar successful movies:
⚠ STRICT RULE: Use these for WRITING STYLE, TONE, and SENTENCE RHYTHM ONLY.
Do NOT borrow any character names, plot events, locations, or story elements.
Every word of your output must describe ONLY the movie in MOVIE DETAILS above.
If you find yourself writing anything not in MOVIE DETAILS, stop and rewrite.
__REFS__

CONSTRAINTS:
- Output ONLY the JSON object (no markdown, no backticks, no commentary).
- tagline must be original and specific to this movie, not generic.
- overview must be written as compelling promo copy, not a plot summary.
- End immediately after the final '}'.

Now output the JSON:
'''

for name in ["prompt_baseline.txt", "prompt_trial.txt"]:
    path = os.path.join(EXP_DIR, name)
    if not os.path.exists(path):
        with open(path, "w") as f:
            f.write(PROMPT_TEMPLATE)
        print(f"Created {path}")
    else:
        print(f"Exists: {path}")

# ── Rubric JSON (same for baseline and trial initially) ──
RUBRIC_DICT = {
    "narrative_fidelity": {
        "description": "Output follows intended premise without adding unintended plot elements",
        "5": "No negative_constraints violated. All content anchored strictly to core_premise and thematic_core.",
        "3": "Minor trope drift or one incidental element not in the premise; no invented characters or major additions.",
        "1": "Invented characters, disappearances, villains, or plot elements explicitly excluded by negative_constraints."
    },
    "genre_alignment": {
        "description": "Output matches the intended genre and tone",
        "5": "Vocabulary, sentence rhythm, and emotional register fully match the target genre.",
        "3": "Mostly aligned but one phrase or image feels like it belongs to a different genre.",
        "1": "Clearly wrong tone."
    },
    "visual_specificity": {
        "description": "Poster direction includes concrete, actionable visual elements",
        "5": "Specifies at least 3 of: subject position, lighting direction, color palette, dominant shadow/silhouette, and one symbolic object. A designer could start sketching immediately from this description.",
        "3": "Mentions atmosphere and mood but missing 2+ of the above specifics.",
        "1": "Only generic descriptors like 'dark and moody' with no actionable detail."
    },
    "creative_specificity": {
        "description": "Output contains distinctive, non-generic phrasing",
        "5": "The tagline contains an unexpected word combination or image that could not have come from a generic LLM prompt. Memorable on first read.",
        "3": "Competent and on-brief but interchangeable with many other films in this genre.",
        "1": "Cliche phrase that appears verbatim in common marketing templates."
    },
    "output_format_validity": {
        "description": "Output follows required structure and word limits",
        "5": "All three fields present. tagline <= 12 words, overview <= 80 words, poster_art_direction <= 60 words.",
        "3": "All fields present but one field slightly exceeds the word limit (<= 20% over).",
        "1": "A required field is missing, or one field is more than 20% over the word limit."
    }
}

for name in ["rubric_baseline.json", "rubric_trial.json"]:
    path = os.path.join(EXP_DIR, name)
    if not os.path.exists(path):
        with open(path, "w") as f:
            json.dump(RUBRIC_DICT, f, indent=2)
        print(f"Created {path}")
    else:
        print(f"Exists: {path}")

print("\n✅ experiment_config/ ready")


## Section 1: Data, RAG Index, Model & Captions
Run all cells in this section once. Takes ~5 min on T4.

In [ ]:
# Cell 1-A: Load TMDB dataset + genre mapping
file_path = os.path.join(REPO_ROOT, "tmdb-fetch/tmdb_movies.csv")
movies_df = pd.read_csv(file_path)
print(f"✅ Loaded {len(movies_df)} movies")

# Genre mapping
def tmdb_get(url, params=None):
    headers = {"Authorization": f"Bearer {TMDB_BEARER}", "Content-Type": "application/json;charset=utf-8"}
    r = requests.get(url, headers=headers, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

genre_data = tmdb_get("https://api.themoviedb.org/3/genre/movie/list", params={"language": "en-US"})
GENRE_ID_TO_NAME = {g["id"]: g["name"] for g in genre_data.get("genres", [])}
GENRE_NAME_TO_ID = {v.lower(): k for k, v in GENRE_ID_TO_NAME.items()}
print(f"✅ Genres loaded: {len(GENRE_ID_TO_NAME)}")

In [ ]:
# Cell 1-B: Build RAG corpus + FAISS index
from sentence_transformers import SentenceTransformer

corpus_df = movies_df[
    movies_df["tagline"].notna() & (movies_df["tagline"].str.strip() != "")
].copy().reset_index(drop=True)
print(f"Corpus size: {len(corpus_df)} (excluded {len(movies_df) - len(corpus_df)} missing tagline)")

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
corpus_texts = (corpus_df["title"] + " " + corpus_df["tagline"] + " " + corpus_df["overview"]).tolist()
embeddings = embed_model.encode(corpus_texts, normalize_embeddings=True, show_progress_bar=True).astype("float32")

faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)
print(f"✅ FAISS index: {faiss_index.ntotal} vectors")

In [ ]:
# Cell 1-C: Load generator (Mistral-7B 4-bit)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, token=HF_TOKEN, device_map="auto", quantization_config=bnb_config,
)
print("✅ Model loaded on:", model.device)

In [ ]:
# Cell 1-D: Load cached poster captions
CAPTION_PATH = os.path.join(REPO_ROOT, "tmdb-fetch/poster_captions.csv")
if os.path.exists(CAPTION_PATH):
    caption_df = pd.read_csv(CAPTION_PATH)
    corpus_df = corpus_df.merge(caption_df[["id", "poster_caption"]], on="id", how="left")
    print(f"✅ Captions loaded. Coverage: {corpus_df['poster_caption'].notna().sum()}/{len(corpus_df)}")
else:
    print("⚠️ poster_captions.csv not found — V2 visual grounding will be unavailable")

## Section 3: Core Pipeline Functions
These are the building blocks. Run all cells, then experiment in Section 5.

In [ ]:
# Cell 3-A: Genre filter helper
def _normalize_genre_filter(genre_filter):
    if genre_filter is None:
        return None
    if isinstance(genre_filter, (list, tuple)) and len(genre_filter) > 0:
        if isinstance(genre_filter[0], str):
            ids = [GENRE_NAME_TO_ID[name.lower()] for name in genre_filter if name.lower() in GENRE_NAME_TO_ID]
            return ids if ids else []
        if isinstance(genre_filter[0], int):
            return list(genre_filter)
    raise ValueError("genre_filter must be None, list of genre name strings, or list of genre int IDs.")

In [ ]:
# Cell 3-B: Retrieval
def retrieve(query: str, k: int = 5, genre_filter=None) -> pd.DataFrame:
    q = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    gf_ids = _normalize_genre_filter(genre_filter)
    if gf_ids is None:
        scores, idx = faiss_index.search(q, k)
        out = corpus_df.iloc[idx[0]].copy()
        out["score"] = scores[0]
        return out.reset_index(drop=True)
    gf_ids = set(gf_ids)
    mask = corpus_df["genre_ids"].apply(lambda gids: any(gid in gf_ids for gid in eval(gids))).to_numpy()
    candidate_idx = np.where(mask)[0]
    if candidate_idx.size == 0:
        scores, idx = faiss_index.search(q, k)
        out = corpus_df.iloc[idx[0]].copy()
        out["score"] = scores[0]
        return out.reset_index(drop=True)
    cand_emb = embeddings[candidate_idx]
    scores = cand_emb @ q[0]
    topk_local = np.argsort(-scores)[:k]
    topk_idx = candidate_idx[topk_local]
    out = corpus_df.iloc[topk_idx].copy()
    out["score"] = scores[topk_local]
    return out.reset_index(drop=True)

In [ ]:
# Cell 3-C: Reference text builder
def refs_to_text(df: pd.DataFrame, n: int = 5, use_visual: bool = True) -> str:
    lines = []
    for _, r in df.head(n).iterrows():
        if pd.notna(r["tagline"]) and r["tagline"].strip():
            ov = (r["overview"] or "")[:200].replace("\n", " ")
            line = f'- {r["title"]}: tagline: "{r["tagline"]}" | overview: "{ov}"'
            if use_visual and "poster_caption" in r and pd.notna(r.get("poster_caption")):
                caption = str(r["poster_caption"]).strip()
                if caption:
                    line += f' | visual style: "{caption[:300]}"'
            lines.append(line)
    return "\n".join(lines)

In [ ]:
# Cell 3-D: Text generation
def generate_text(prompt: str, max_new_tokens: int = 320, temperature: float = 0.6) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=temperature, top_p=0.9,
            repetition_penalty=1.10,
            pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
# Cell 3-E: JSON extraction
REQUIRED_KEYS = {"overview", "tagline", "poster_art_direction"}

def extract_best_json(text: str) -> dict:
    decoder = json.JSONDecoder()
    objs = []
    i = 0
    while True:
        start = text.find("{", i)
        if start == -1:
            break
        try:
            obj, end = decoder.raw_decode(text[start:])
            objs.append(obj)
            i = start + end
        except json.JSONDecodeError:
            i = start + 1
    if not objs:
        raise ValueError("No JSON object found in model output.")
    def score(obj):
        if not isinstance(obj, dict): return -10
        if not REQUIRED_KEYS.issubset(set(obj.keys())): return -5
        s = 0
        tl = obj.get("tagline", "")
        if isinstance(tl, str) and len(tl.strip()) > 0: s += 2
        if isinstance(tl, str) and len(tl.split()) <= 14: s += 1
        if isinstance(obj.get("poster_art_direction",""), str) and len(obj.get("poster_art_direction","").strip()) > 20: s += 2
        if isinstance(obj.get("overview",""), str) and len(obj.get("overview","").strip()) > 40: s += 2
        return s
    return max(objs, key=score)

### 🔧 Prompt templates — edit files under `experiment_config/`

**Do not paste long prompts here.** Edit **`prompt_baseline.txt`** (frozen reference) and **`prompt_trial.txt`** (experiments).

Placeholders in those files (literal strings): `__PREMISE__`, `__THEME__`, `__NEGATIVES__`, `__REFS__`

**Cell 3-F** below loads them and defines `build_prompt_baseline`, `build_prompt_trial`, and active `build_prompt` (defaults to **trial** — switch before Section 6 if you freeze baseline).

**Rubrics:** JSON files **`rubric_baseline.json`** / **`rubric_trial.json`** (Cell 4-A). Baseline vs trial must keep the **same dimension keys** so scores are comparable.


In [ ]:
# Cell 3-F: Load prompt templates from experiment_config/

def experiment_config_dir():
    # Prefer configs inside the cloned repo; fall back to cwd (local dev).
    candidates = []
    try:
        candidates.append(os.path.join(REPO_ROOT, "experiment_config"))
    except NameError:
        pass
    candidates.append(os.path.join(os.getcwd(), "experiment_config"))
    for c in candidates:
        if c and os.path.isfile(os.path.join(c, "prompt_baseline.txt")):
            return c
    raise FileNotFoundError(
        "experiment_config/ not found. Add it next to the notebook or inside REPO_ROOT "
        "(copy from the course repo so Colab sees it after clone)."
    )


CONFIG_DIR = experiment_config_dir()


def _fill_prompt_template(template: str, movie_input: dict, refs: str) -> str:
    return (
        template.replace("__PREMISE__", movie_input.get("core_premise", "") or "")
        .replace("__THEME__", movie_input.get("thematic_core", "") or "")
        .replace("__NEGATIVES__", movie_input.get("negative_constraints", "") or "")
        .replace("__REFS__", refs or "")
    )


def _make_prompt_builder(path: str):
    def _fn(movie_input: dict, refs: str) -> str:
        with open(path, encoding="utf-8") as f:
            body = f.read()
        return _fill_prompt_template(body, movie_input, refs)

    return _fn


build_prompt_baseline = _make_prompt_builder(os.path.join(CONFIG_DIR, "prompt_baseline.txt"))
build_prompt_trial = _make_prompt_builder(os.path.join(CONFIG_DIR, "prompt_trial.txt"))

# Active builder for run_v1_v2 / run_v3 / Section 6 unless you override
build_prompt = build_prompt_trial

print(f"✅ Prompt loaders ready → CONFIG_DIR={CONFIG_DIR}")


In [ ]:
# Cell 3-G: V1 / V2 runner
def run_v1_v2(
    movie_input: dict,
    k: int = 5,
    genre_filter=None,
    temperature: float = 0.6,
    prompt_builder=None,
):
    pb = prompt_builder or build_prompt
    query = movie_input["core_premise"]
    p1 = pb(movie_input, refs="(none)")
    t1 = generate_text(p1, temperature=temperature)
    j1 = extract_best_json(t1)
    topk = retrieve(query, k=k, genre_filter=genre_filter)
    refs = refs_to_text(topk, n=k, use_visual=True)
    p2 = pb(movie_input, refs=refs)
    t2 = generate_text(p2, temperature=temperature)
    j2 = extract_best_json(t2)
    return j1, j2, topk, refs

print("✅ run_v1_v2 ready")


In [ ]:
# Cell 3-H: V3 Agentic Loop runner
from openai import OpenAI


def run_v3(
    movie_input,
    k=5,
    genre_filter=None,
    temperature=0.6,
    max_iter=3,
    judge_client=None,
    prompt_builder=None,
    rubric=None,
):
    pb = prompt_builder or build_prompt
    rb = rubric if rubric is not None else RUBRIC
    dim_order = list(rb.keys())

    if judge_client is None:
        judge_client = OpenAI()

    query = movie_input["core_premise"]
    topk = retrieve(query, k=k, genre_filter=genre_filter)
    refs = refs_to_text(topk, n=k, use_visual=True)
    feedback = ""
    history = []
    best_assets = None
    for iteration in range(1, max_iter + 1):
        base = pb(movie_input, refs=refs)
        prompt = (
            base + f"\n\nPREVIOUS FEEDBACK:\n{feedback}\nAddress the above before regenerating.\n"
            if feedback
            else base
        )
        raw = generate_text(prompt, temperature=temperature)
        try:
            assets = extract_best_json(raw)
        except (ValueError, TypeError, json.JSONDecodeError):
            history.append(
                {"iter": iteration, "assets": None, "scores": None, "average": None, "passed": False}
            )
            feedback = "Your previous output was not valid JSON with the required keys. Output exactly one JSON object with keys overview, tagline, poster_art_direction."
            continue
        best_assets = assets
        result = llm_as_judge(
            movie_input, assets, client=judge_client, verbose=False, rubric=rb
        )
        scores = result.get("scores") or {}
        avg = result.get("average")
        dim_scores = []
        for dim in dim_order:
            entry = scores.get(dim, {}) if isinstance(scores, dict) else {}
            s = entry.get("score")
            if isinstance(s, (int, float)):
                dim_scores.append(s)
        passed = len(dim_scores) == len(dim_order) and all(s >= 3 for s in dim_scores)
        history.append(
            {"iter": iteration, "assets": assets, "scores": scores, "average": avg, "passed": passed}
        )
        tagline = (assets.get("tagline") or "")[:120]
        avg_str = f"{avg:.2f}" if isinstance(avg, (int, float)) else "n/a"
        print(f"[v3 iter {iteration}] tagline={tagline!r} average={avg_str}")
        if passed:
            break
        low_dim, low_score = None, float("inf")
        for dim in dim_order:
            entry = scores.get(dim, {}) if isinstance(scores, dict) else {}
            s = entry.get("score")
            if isinstance(s, (int, float)) and s < low_score:
                low_score, low_dim = s, dim
        feedback = (
            scores.get(low_dim, {}).get("reason", f"Improve the {low_dim} dimension.")
            if low_dim
            else "Improve the output on all rubric dimensions."
        )
    return best_assets, history


print("✅ run_v3 ready")


## Section 4: Evaluation Functions

### 🔧 `RUBRIC` — JSON files in `experiment_config/`

**Cell ID: 4-A** loads **`rubric_baseline.json`** + **`rubric_trial.json`**. Edit those files for A/B rubric work; keep the same keys in both files.

The active default for `llm_as_judge(rubric=None)` is **`RUBRIC_TRIAL`** (change the assignment in 4-A if you want baseline as default).


In [ ]:
# Cell 4-A: Load rubrics from experiment_config/*.json

from openai import OpenAI

JUDGE_MODEL = "gpt-4o"


def _load_rubric(name: str) -> dict:
    path = os.path.join(CONFIG_DIR, f"rubric_{name}.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


RUBRIC_BASELINE = _load_rubric("baseline")
RUBRIC_TRIAL = _load_rubric("trial")

# Default active rubric (matches build_prompt default = trial)
RUBRIC = RUBRIC_TRIAL

DIMENSIONS = list(RUBRIC.keys())
assert set(RUBRIC_BASELINE.keys()) == set(
    RUBRIC_TRIAL.keys()
), "baseline and trial rubrics must use the same dimension keys"

print(f"✅ Rubrics loaded from {CONFIG_DIR}")


In [ ]:
# Cell 4-B: Judge prompt builder


def _build_judge_prompt(movie_input: dict, generated_output: dict, rubric=None):
    rubric = rubric if rubric is not None else RUBRIC
    rubric_text = ""
    for dim, content in rubric.items():
        rubric_text += f"\n{dim}:\n"
        rubric_text += f"  5 = {content['5']}\n"
        rubric_text += f"  3 = {content['3']}\n"
        rubric_text += f"  1 = {content['1']}\n"

    dim_list = list(rubric.keys())
    schema_inner = ",\n".join(
        [f'    "{d}":      {{"score": <int>, "reason": "<str>"}}' for d in dim_list]
    )

    return f"""You are an expert film marketing evaluator.
Score the following generated marketing assets on {len(dim_list)} dimensions, each on a 1–5 integer scale.

[INPUT CONCEPT]
core_premise: {movie_input.get('core_premise', '')}
thematic_core: {movie_input.get('thematic_core', '')}
negative_constraints: {movie_input.get('negative_constraints', '')}

[GENERATED OUTPUTS]
tagline: {generated_output.get('tagline', '')}
overview: {generated_output.get('overview', '')}
poster_art_direction: {generated_output.get('poster_art_direction', '')}

[RUBRIC]{rubric_text}
Scoring rules:
- Score each dimension independently.
- Assign integer scores only: 1, 2, 3, 4, or 5.
- Write one concise sentence as reason for each score.
- Do NOT copy phrasing from the generated output in your reasons.

Return ONLY valid JSON. No markdown, no preamble. Schema:
{{
  "scores": {{
{schema_inner}
  }},
  "average": <float>
}}"""


print("✅ _build_judge_prompt ready")


In [ ]:
# Cell 4-C: llm_as_judge() — single evaluation


def llm_as_judge(
    movie_input,
    generated_output,
    client=None,
    max_retries=3,
    retry_delay=2.0,
    verbose=True,
    rubric=None,
):
    rubric = rubric if rubric is not None else RUBRIC
    dims = list(rubric.keys())
    if client is None:
        client = OpenAI()
    prompt = _build_judge_prompt(movie_input, generated_output, rubric=rubric)
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=JUDGE_MODEL,
                max_tokens=1000,
                response_format={"type": "json_object"},
                messages=[
                    {
                        "role": "system",
                        "content": "You are a strict film marketing evaluator. Return only valid JSON matching the requested schema.",
                    },
                    {"role": "user", "content": prompt},
                ],
            )
            raw_text = response.choices[0].message.content.strip()
            raw_text = re.sub(r"^```(?:json)?\s*", "", raw_text)
            raw_text = re.sub(r"\s*```$", "", raw_text)
            result = json.loads(raw_text)
            scores = result.get("scores", {})
            vals = [
                v["score"]
                for v in scores.values()
                if isinstance(v.get("score"), (int, float))
            ]
            result["average"] = round(sum(vals) / len(vals), 2) if vals else 0.0
            result["error"] = None
            if verbose:
                print(
                    f"  avg={result['average']:.2f}  "
                    + "  ".join(f"{d}={scores.get(d, {}).get('score', '?')}" for d in dims)
                )
            return result
        except (json.JSONDecodeError, KeyError) as e:
            if attempt < max_retries:
                time.sleep(retry_delay)
            else:
                return {"scores": {}, "average": 0.0, "error": str(e)}


def summarize_batch(batch_results, dimensions=None):
    dims = dimensions if dimensions is not None else DIMENSIONS
    per_dim = {d: [] for d in dims}
    averages = []
    error_count = 0
    for record in batch_results:
        r = record["result"]
        if r.get("error"):
            error_count += 1
            continue
        averages.append(r["average"])
        for dim in dims:
            score = r["scores"].get(dim, {}).get("score")
            if score is not None:
                per_dim[dim].append(score)
    summary = {
        "n": len(batch_results),
        "error_count": error_count,
        "overall_mean": round(sum(averages) / len(averages), 2) if averages else 0.0,
        "per_dimension": {},
    }
    for dim, vals in per_dim.items():
        if vals:
            summary["per_dimension"][dim] = {
                "mean": round(sum(vals) / len(vals), 2),
                "min": min(vals),
                "max": max(vals),
            }
    return summary


print("✅ llm_as_judge + summarize_batch ready")


---
## Section 5: 🧪 EXPERIMENT ZONE

**This is where you iterate.** Everything above loads infrastructure from **`experiment_config/`** — edit the `.txt` / `.json` files there, then re-run **Cell 3-F** and **Cells 4-A–B**.

Section order: **5B → 5A → 5C → 5D** (5D = side-by-side **baseline vs trial** on one idea).

### Experiments at a glance

| Step | What | Source files | Time |
|------|------|----------------|------|
| **5B** | Val subset regeneration + judge | `prompt_*.txt`, `rubric_*.json` (via 3-F / 4-A) | ~1 hr |
| **5A** | Re-score cached outputs (no GPU) | `rubric_*.json` | ~30 min |
| **5C** | One idea V1/V2/V3 | same files | ~5 min |
| **5D** | **Compare baseline vs trial** (same refs) | all four files | ~5–10 min |

### ⚠️ Val set only (n=145). Test set is FROZEN until Section 6.


### 5B: Prompt Improvement — Regenerate on val subset

**Edit files:** `experiment_config/prompt_trial.txt` (and optionally `prompt_baseline.txt`). Then **re-run Cell 3-F**, then **Cells 4-A–B** if you changed rubrics.

**What this does:** Runs V1 and V2 on a small val subset using **`build_prompt`** (= **trial** template by default) and **`RUBRIC`** (= **trial** rubric by default).

**How to use:**
1. Edit **`prompt_trial.txt`** / **`rubric_trial.json`** as needed
2. Re-run **Cell 3-F**, **4-A**, **4-B**
3. Run the **code cell below**

**Compare runs:** **Section 5D** shows baseline vs trial on one idea; **5B** here is still “trial pipeline only” over many val rows — note numbers if you need history.

**Key question:** Does V2 (RAG) beat V1 clearly enough to matter?


In [ ]:
# Cell 5B: Run V1/V2 on val subset with current prompt & rubric
N_VAL_SAMPLE = 10  # Start small, increase once you see promising results

QA_VAL_PATH = os.path.join(CACHE_DIR, "qa_val.json")
with open(QA_VAL_PATH, "r") as f:
    qa_val = json.load(f)

# Use best config from tuning
BEST_K = 3
BEST_GF = False  # genre_filter=False won tuning
BEST_TEMP = 0.6

val_sample = qa_val[:N_VAL_SAMPLE]
judge_client = OpenAI()
results_v1, results_v2 = [], []

for i, rec in enumerate(val_sample, 1):
    movie_input = {
        "core_premise": rec["core_premise"],
        "thematic_core": rec["thematic_core"],
        "negative_constraints": rec["negative_constraints"],
    }
    print(f"\n[{i}/{N_VAL_SAMPLE}] Generating & judging...")
    v1, v2, topk, refs = run_v1_v2(movie_input, k=BEST_K, genre_filter=None, temperature=BEST_TEMP)

    j1 = llm_as_judge(movie_input, v1, client=judge_client, verbose=False)
    j2 = llm_as_judge(movie_input, v2, client=judge_client, verbose=False)
    results_v1.append({"id": rec.get("movie_id", i), "result": j1})
    results_v2.append({"id": rec.get("movie_id", i), "result": j2})

    print(f"  V1={j1['average']:.2f}  V2={j2['average']:.2f}  delta={j2['average']-j1['average']:+.2f}")
    time.sleep(0.5)

s1 = summarize_batch(results_v1)
s2 = summarize_batch(results_v2)
print(f"\n{'='*50}")
print(f"V1 overall: {s1['overall_mean']:.2f}")
print(f"V2 overall: {s2['overall_mean']:.2f}")
print(f"Delta: {s2['overall_mean'] - s1['overall_mean']:+.2f}")
print(f"{'='*50}")
for dim in DIMENSIONS:
    d1 = s1['per_dimension'].get(dim, {}).get('mean', 0)
    d2 = s2['per_dimension'].get(dim, {}).get('mean', 0)
    print(f"  {dim:<28} V1={d1:.2f}  V2={d2:.2f}  delta={d2-d1:+.2f}")

### 5A: Rubric Tightening — Re-judge existing outputs

**Edit files:** `experiment_config/rubric_baseline.json` / `rubric_trial.json`, then **re-run Cells 4-A & 4-B**.

**What this does:** Loads cached evaluation JSON and re-scores it with the loaded rubrics — **no** new Mistral generation.

**How to use:**
1. Edit rubric JSON files
2. Re-run **Cells 4-A** and **4-B**
3. Run the **code cell below**

**Compare runs:** Note aggregate scores, change rubrics, re-run — or use **5D** to see baseline vs trial judge side-by-side on fresh generations.

**Why this is cheap:** Judge API only.

**Note:** The code skeleton below still has TODOs for reading `assets` from your cache JSON — wire it to your `test_results*.json` format when ready.


In [ ]:
# Cell 5A: Re-judge cached outputs with current (trial) rubric
# No generation needed — just re-score existing outputs with the active RUBRIC.
# Works with test_results.json (from previous Cell 9-E) or test_results_final.json (from Section 6).

REJUDGE_SOURCE = os.path.join(CACHE_DIR, "test_results.json")
if not os.path.isfile(REJUDGE_SOURCE):
    REJUDGE_SOURCE = os.path.join(CACHE_DIR, "test_results_final.json")

if not os.path.isfile(REJUDGE_SOURCE):
    print(f"⚠️ No cached results found. Run Section 5C or 6 first to generate outputs.")
else:
    with open(REJUDGE_SOURCE, "r") as f:
        cached = json.load(f)

    # Load QA inputs to pair with cached outputs
    qa_source = os.path.join(CACHE_DIR, "qa_test.json")
    with open(qa_source, "r") as f:
        qa_records = json.load(f)
    qa_by_id = {int(r["movie_id"]): r for r in qa_records}

    judge_client = OpenAI()
    n_rejudge = min(20, len(cached["v1"]))

    rejudge_v1, rejudge_v2, rejudge_v3 = [], [], []

    for i in range(n_rejudge):
        for variant_key, result_list in [("v1", rejudge_v1), ("v2", rejudge_v2), ("v3", rejudge_v3)]:
            rec = cached[variant_key][i]
            mid = rec["id"]
            qa = qa_by_id.get(mid)
            if not qa:
                continue
            movie_input = {
                "core_premise": qa["core_premise"],
                "thematic_core": qa["thematic_core"],
                "negative_constraints": qa["negative_constraints"],
            }
            # Extract assets — Section 6 saves them under "assets" key
            assets = rec.get("assets")
            if assets is None:
                # Fallback: try to extract from result structure (older format)
                print(f"⚠️ No 'assets' key for {variant_key}[{i}] (id={mid}). Skipping.")
                continue

            j = llm_as_judge(movie_input, assets, client=judge_client, verbose=False)
            result_list.append({"id": mid, "result": j})

        if (i + 1) % 5 == 0:
            print(f"  Re-judged {i+1}/{n_rejudge} records...")

    print(f"\nRe-judged {n_rejudge} records with current RUBRIC (trial).")
    print(f"Compare with original scores in cached file.\n")

    for label, results in [("V1", rejudge_v1), ("V2", rejudge_v2), ("V3", rejudge_v3)]:
        if results:
            s = summarize_batch(results)
            print(f"{label}: overall={s['overall_mean']:.2f}  " +
                  "  ".join(f"{d}={s['per_dimension'].get(d,{}).get('mean',0):.2f}" for d in DIMENSIONS))

    # Show original scores for comparison
    if "summary" in cached:
        print(f"\n--- Original scores (from cache) ---")
        for vk in ["v1", "v2", "v3"]:
            s = cached["summary"][vk]
            print(f"{vk.upper()}: overall={s['overall_mean']:.2f}  " +
                  "  ".join(f"{d}={s['per_dimension'].get(d,{}).get('mean',0):.2f}" for d in list(cached["summary"]["v1"]["per_dimension"].keys())))


### 5C: Sanity Check — Eyeball one idea

Uses whatever is currently in **Cell 3-F** and **Cell 4-A** — edit those first, then run this cell.

**What this does:** One synthetic idea through **V1 → V2 → V3** so you can read JSON before spending val-set time.

**When to use:** After prompt or rubric edits, check that outputs look sane.


In [ ]:
# Cell 5C: Quick sanity check — one idea, full V1/V2/V3

test_idea = {
    "core_premise": "A man wakes up in a hospital with no memory of his past. "
                    "People claiming to be his family and friends begin to appear one by one — "
                    "each with a different account of who he was.",
    "thematic_core": "Identity is not what you remember — it is what others need you to believe.",
    "negative_constraints": "Do NOT reveal whether the visitors are real or fabricated. "
                            "Do NOT use action sequences or physical violence as plot drivers.",
}

print("=" * 60)
print("Running V1 (zero-shot) and V2 (RAG+visual)...")
v1, v2, topk, refs = run_v1_v2(test_idea, k=3, temperature=0.6)

print("\n--- V1 (Zero-Shot) ---")
print(json.dumps(v1, indent=2))
print("\n--- V2 (RAG+Visual) ---")
print(json.dumps(v2, indent=2))

print("\nRunning V3 (Agentic)...")
judge_client = OpenAI()
v3, v3_history = run_v3(test_idea, k=3, temperature=0.6, judge_client=judge_client)
print("\n--- V3 (Agentic) ---")
print(json.dumps(v3, indent=2))

print("\n--- Judge all three ---")
for label, output in [("V1", v1), ("V2", v2), ("V3", v3)]:
    print(f"\n{label}:")
    llm_as_judge(test_idea, output, client=judge_client, verbose=True)


### 5D: Baseline vs trial — same idea, compared

**Purpose:** Run **baseline prompt + baseline rubric** and **trial prompt + trial rubric** on the **same** synthetic idea and **same retrieved refs** for V2. You get a small table instead of “only the last experiment.”

**Cross-judge rows:** same V2 JSON, swapped rubric — separates “prompt changed” vs “scoring criteria changed.”

**Cost:** 4× Mistral (V1+V2 ×2 configs) + several GPT-4o judge calls.


In [ ]:
# Cell 5D: Compare baseline vs trial configs (files in experiment_config/)

from IPython.display import display


def compare_baseline_vs_trial_one_idea(movie_input: dict, k: int = 3, temperature: float = 0.6):
    jc = OpenAI()
    query = movie_input["core_premise"]
    topk = retrieve(query, k=k, genre_filter=None)
    refs = refs_to_text(topk, n=k, use_visual=True)

    rows = []
    assets_store = {}

    configs = [
        ("baseline", build_prompt_baseline, RUBRIC_BASELINE),
        ("trial", build_prompt_trial, RUBRIC_TRIAL),
    ]

    for label, pb, rb in configs:
        p1 = pb(movie_input, refs="(none)")
        t1 = generate_text(p1, temperature=temperature)
        j1 = extract_best_json(t1)
        p2 = pb(movie_input, refs=refs)
        t2 = generate_text(p2, temperature=temperature)
        j2 = extract_best_json(t2)
        assets_store[label] = {"v1": j1, "v2": j2}

        r1 = llm_as_judge(movie_input, j1, client=jc, verbose=False, rubric=rb)
        r2 = llm_as_judge(movie_input, j2, client=jc, verbose=False, rubric=rb)
        rows.append({"config": label, "stage": "V1", "average": r1.get("average")})
        rows.append({"config": label, "stage": "V2", "average": r2.get("average")})

    df_main = pd.DataFrame(rows).pivot(index="stage", columns="config", values="average")
    print("\n=== Averages (native prompt + rubric per column) ===")
    display(df_main)

    j2_base = assets_store["baseline"]["v2"]
    j2_trial = assets_store["trial"]["v2"]
    cross = pd.DataFrame(
        [
            {
                "same_text": "baseline V2 JSON",
                "rubric_used": "trial",
                "average": llm_as_judge(
                    movie_input, j2_base, client=jc, verbose=False, rubric=RUBRIC_TRIAL
                ).get("average"),
            },
            {
                "same_text": "trial V2 JSON",
                "rubric_used": "baseline",
                "average": llm_as_judge(
                    movie_input, j2_trial, client=jc, verbose=False, rubric=RUBRIC_BASELINE
                ).get("average"),
            },
        ]
    )
    print("\n=== Cross-judge V2 only (swap rubric on fixed JSON) ===")
    display(cross)

    return assets_store, df_main, cross


compare_idea = {
    "core_premise": "A man wakes up in a hospital with no memory of his past. "
    "People claiming to be his family and friends begin to appear one by one — "
    "each with a different account of who he was.",
    "thematic_core": "Identity is not what you remember — it is what others need you to believe.",
    "negative_constraints": "Do NOT reveal whether the visitors are real or fabricated. "
    "Do NOT use action sequences or physical violence as plot drivers.",
}

compare_baseline_vs_trial_one_idea(compare_idea, k=3, temperature=0.6)


### 5E: Statistical Significance Test (Analysis A)

**What this does:** Runs a Wilcoxon signed-rank test on V1 vs V2 per-record overall scores from the cached test set results.

**Why:** The test set shows V2 avg=4.46 vs V1 avg=4.42 — a difference of 0.04. This test determines if that difference is statistically significant or could be due to chance.

**When to run:** Anytime after test_results.json or test_results_final.json exists. No generation needed.

**Output for paper:** Report the p-value in §5.7, even if p > 0.05 (honest reporting).


In [ ]:
# Cell 5E: Wilcoxon signed-rank test — V1 vs V2 per-record scores
from scipy.stats import wilcoxon

STAT_SOURCE = os.path.join(CACHE_DIR, "test_results_final.json")
if not os.path.isfile(STAT_SOURCE):
    STAT_SOURCE = os.path.join(CACHE_DIR, "test_results.json")

if not os.path.isfile(STAT_SOURCE):
    print("⚠️ No test results found. Run Section 6 or use existing test_results.json.")
else:
    with open(STAT_SOURCE, "r") as f:
        cached = json.load(f)

    v1_scores = [r["result"]["average"] for r in cached["v1"] if not r["result"].get("error")]
    v2_scores = [r["result"]["average"] for r in cached["v2"] if not r["result"].get("error")]
    v3_scores = [r["result"]["average"] for r in cached["v3"] if not r["result"].get("error")]

    n = min(len(v1_scores), len(v2_scores))
    v1_scores, v2_scores = v1_scores[:n], v2_scores[:n]

    print(f"Records: n={n}")
    print(f"V1 mean: {np.mean(v1_scores):.3f}")
    print(f"V2 mean: {np.mean(v2_scores):.3f}")
    print(f"Delta:   {np.mean(v2_scores) - np.mean(v1_scores):+.3f}")
    print()

    # V1 vs V2
    if all(a == b for a, b in zip(v1_scores, v2_scores)):
        print("V1 vs V2: All scores identical — no test needed.")
    else:
        stat, p = wilcoxon(v1_scores, v2_scores)
        print(f"V1 vs V2: Wilcoxon statistic={stat:.1f}, p={p:.4f}")
        if p < 0.05:
            print("  → Statistically significant at α=0.05")
        else:
            print(f"  → NOT significant (p={p:.4f}). Report as: 'directional improvement, not statistically significant at n={n}'")

    # V2 vs V3
    n23 = min(len(v2_scores), len(v3_scores))
    v3_scores = v3_scores[:n23]
    v2_sub = v2_scores[:n23]
    print()
    if all(a == b for a, b in zip(v2_sub, v3_scores)):
        print("V2 vs V3: All scores identical — no test needed.")
    else:
        stat, p = wilcoxon(v2_sub, v3_scores)
        print(f"V2 vs V3: Wilcoxon statistic={stat:.1f}, p={p:.4f}")

    # Per-dimension breakdown for visual specificity (key dimension)
    print("\n--- Per-dimension: Visual Specificity ---")
    v1_vis = [r["result"]["scores"].get("visual_specificity", {}).get("score", 0)
              for r in cached["v1"][:n] if not r["result"].get("error")]
    v2_vis = [r["result"]["scores"].get("visual_specificity", {}).get("score", 0)
              for r in cached["v2"][:n] if not r["result"].get("error")]
    if not all(a == b for a, b in zip(v1_vis, v2_vis)):
        stat, p = wilcoxon(v1_vis, v2_vis)
        print(f"Visual spec V1 vs V2: stat={stat:.1f}, p={p:.4f}")
    else:
        print("Visual spec: identical scores")


---
## Section 6: Final Test Set Evaluation

### ⚠️ RUN THIS ONLY ONCE — after all experiments in Section 5 are complete.

This runs V1/V2/V3 on the frozen test set (n=36) with the final prompt and rubric.
Results are saved to `to_be_cached/test_results_final.json`.

**Before running, confirm:**
- [ ] `build_prompt()` is finalized (Cell 3-F)
- [ ] `RUBRIC` is finalized (Cell 4-A)
- [ ] Section 5 experiments showed V2 > V1 (or you're ready to report whatever the result is)

In [ ]:
# Cell 6: Final evaluation on test set
# ⚠️ FROZEN — run only after Section 5 experiments are complete

QA_TEST_PATH = os.path.join(CACHE_DIR, "qa_test.json")
FINAL_RESULTS_CACHE = os.path.join(CACHE_DIR, "test_results_final.json")

with open(QA_TEST_PATH, "r") as f:
    qa_test = json.load(f)

print(f"Test set: {len(qa_test)} records")
print(f"Config: k={BEST_K}, genre_filter={BEST_GF}, temp={BEST_TEMP}")
print(f"Saving to: {FINAL_RESULTS_CACHE}")

if os.path.isfile(FINAL_RESULTS_CACHE):
    print(f"\n⚠️ {FINAL_RESULTS_CACHE} already exists. Delete it to re-run.")
else:
    judge_client = OpenAI()
    test_v1, test_v2, test_v3 = [], [], []

    for i, rec in enumerate(qa_test, 1):
        print(f"[{i}/{len(qa_test)}] Evaluating...")
        movie_input = {
            "core_premise": rec["core_premise"],
            "thematic_core": rec["thematic_core"],
            "negative_constraints": rec["negative_constraints"],
        }
        v1, v2, _, _ = run_v1_v2(movie_input, k=BEST_K, genre_filter=None, temperature=BEST_TEMP)
        v3, _ = run_v3(movie_input, k=BEST_K, genre_filter=None, temperature=BEST_TEMP, judge_client=judge_client)

        j1 = llm_as_judge(movie_input, v1, client=judge_client, verbose=False)
        j2 = llm_as_judge(movie_input, v2, client=judge_client, verbose=False)
        j3 = llm_as_judge(movie_input, v3, client=judge_client, verbose=False)

        test_v1.append({"id": int(rec["movie_id"]), "result": j1, "assets": v1})
        test_v2.append({"id": int(rec["movie_id"]), "result": j2, "assets": v2})
        test_v3.append({"id": int(rec["movie_id"]), "result": j3, "assets": v3})
        time.sleep(0.5)

    s1, s2, s3 = summarize_batch(test_v1), summarize_batch(test_v2), summarize_batch(test_v3)

    with open(FINAL_RESULTS_CACHE, "w") as f:
        json.dump({"v1": test_v1, "v2": test_v2, "v3": test_v3,
                   "summary": {"v1": s1, "v2": s2, "v3": s3}}, f, indent=2)
    print(f"\n✅ Saved to {FINAL_RESULTS_CACHE}")

    print(f"\n{'='*60}")
    print(f"V1: {s1['overall_mean']:.2f}  V2: {s2['overall_mean']:.2f}  V3: {s3['overall_mean']:.2f}")
    for dim in DIMENSIONS:
        d1 = s1['per_dimension'].get(dim,{}).get('mean',0)
        d2 = s2['per_dimension'].get(dim,{}).get('mean',0)
        d3 = s3['per_dimension'].get(dim,{}).get('mean',0)
        print(f"  {dim:<28} V1={d1:.2f}  V2={d2:.2f}  V3={d3:.2f}")